# Распознаём изображения: исследуем оценки

Этот ноутбук дополняет § 3.1 числовыми упражнениями. NumPy нужен для расчётов; скачивать модель не требуется. Для первого запуска модели используйте приложенные `data/coffee.png` и `data/coffee-crop.png`, а затем повторите опыт на собственных разрешённых снимках по листингу статьи.

## Два режима работы

Числовые упражнения запускаются без весов. Чтобы проверить готовую сеть, установите пакеты из `requirements.txt` со страницы главы и распакуйте `images.zip` рядом с ноутбуком: в папке `data/` находятся тестовые фотографии. В Colab выполните `%pip install -r requirements.txt`. Затем установите `run_model = True` и запустите ячейки сверху вниз. Первый запуск скачивает веса модели; на CPU вычисления могут занять несколько минут. После проверки приложенных снимков замените путь на собственные фотографии.

## Читаем top-3
Измените логиты и проверьте порядок классов. Почему метка вне списка никогда не появится?

In [ ]:
import numpy as np
labels = np.array(["чашка", "принтер", "кот", "книга"])
logits = np.array([2.0, 1.0, 0.0, -1.0])
shifted = logits - logits.max()
prob = np.exp(shifted) / np.exp(shifted).sum()
order = np.argsort(-prob)[:3]
print(list(zip(labels[order], prob[order].round(3))))
assert np.isclose(prob.sum(), 1)

## Проверяем чувствительность к изменению выхода
Измените только второй логит. Сравните лидера и разрыв между первыми двумя оценками.

In [ ]:
changed = logits.copy()
changed[1] += 1.5
p2 = np.exp(changed - changed.max())
p2 /= p2.sum()
print("Было:", labels[np.argmax(prob)], "стало:", labels[np.argmax(p2)])
print("Разрыв:", round(float(np.sort(p2)[-1] - np.sort(p2)[-2]), 3))

## Своя проверка
Запишите top-3 для трёх вариантов фотографии в статье. Впишите наблюдения в таблицу ниже в Markdown-ячейке; код не подставит правильный класс автоматически.

In [ ]:
observations = []  # добавьте словари: файл, top3, что видно на фото
print("Записано наблюдений:", len(observations))

## Опыт с готовой сетью

Сначала выполните модель на `data/coffee.png`, затем на `data/coffee-crop.png` — это полный кадр и обрезка одного исходного изображения. Сравните top-3 и посмотрите на подготовленный вход в листинге статьи. После этого возьмите три собственных разрешённых снимка предмета: на ровном фоне, среди других вещей и крупным планом. До запуска предположите, где ответ будет точнее. Установите зависимости, замените имена файлов и включите `run_model`.

In [ ]:
run_model = False
if run_model:
    from pathlib import Path
    from PIL import Image
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    from transformers import AutoImageProcessor, AutoModelForImageClassification

    files = ["data/coffee.png", "data/coffee-crop.png", "data/chelsea.png"]
    # Правило принято до запуска: узнаём чашку с кофе или кошку,
    # принимая перечисленные метки ImageNet, а не любую посуду или животное.
    accepted = [{504, 967, 968}, {504, 967, 968}, {281, 282, 285}]
    model_id = "microsoft/resnet-18"
    revision = "65a5785d9156231087c481e0c7dd33a5ff6f7e3e"
    processor = AutoImageProcessor.from_pretrained(
        model_id, revision=revision, use_fast=False
    )
    model = AutoModelForImageClassification.from_pretrained(model_id, revision=revision).eval()
    rows = []
    for name, allowed in zip(files, accepted):
        image = Image.open(name).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        with torch.inference_mode():
            logits = model(**inputs).logits[0]
        values, indices = logits.softmax(0).topk(3)
        ids = indices.tolist()
        rows.append({"file": name,
                     "top3": [model.config.id2label[i] for i in ids],
                     "scores": values.tolist(),
                     "correct_top1": ids[0] in allowed,
                     "correct_top3": bool(set(ids) & allowed)})
    report = pd.DataFrame(rows)
    print(report.to_string(index=False))
    # Обращаем нормализацию, чтобы увидеть подготовленный вход.
    image = Image.open(files[0]).convert("RGB")
    pixels = processor(images=image, return_tensors="pt")["pixel_values"][0]
    mean = torch.tensor(processor.image_mean)[:, None, None]
    std = torch.tensor(processor.image_std)[:, None, None]
    prepared = (pixels * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(image); axes[0].set_title("Исходный кадр")
    axes[1].imshow(prepared); axes[1].set_title("Вход после подготовки")
    for ax in axes: ax.axis("off")
    plt.show()


## Разбор результата

Для каждого снимка запишите top-3, оценку первого класса, виден ли предмет целиком и есть ли отвлекающий фон. Сначала сравните полный кадр и обрезку, затем — собственные фотографии. Проверьте правильную метку по изображению и словарю ImageNet. Если предмета нет среди классов, модель всё равно вернёт метки из своего словаря; зафиксируйте это как ограничение задачи, а не как найденное название.

In [ ]:
if run_model:
    print("top-1:", report["correct_top1"].mean(),
          "top-3:", report["correct_top3"].mean())
    print("Это доли для принятого правила меток; две чашки — обрезки одного кадра.")


In [ ]:
if run_model:
    from importlib.metadata import version
    print("Библиотеки:", {name: version(name) for name in
          ["torch", "torchvision", "transformers", "numpy", "pandas"]})
    print("Модель и ревизия:", model_id, revision)


## Самопроверка




### 1. Классификатор вернул для фотографии оценки 0,62, 0,25 и 0,13. Сколько классов он оценил в этом примере?

A. Один.

B. Три.

C. Два.

D. Нельзя определить по оценкам.




### 2. Модель выдала логиты для классов A, B и C: 2, 1 и 0. Какой класс займёт первое место после softmax?

A. C.

B. A и B разделят первое место.

C. A.

D. По логитам нельзя определить порядок.




### 3. Для пяти размеченных снимков верная метка оказалась первой у трёх, второй — у одного, а на одном не попала в top-3. Чему равна top-3 accuracy?

A. $4/5=0{,}8$.

B. $3/5=0{,}6$.

C. $1/5=0{,}2$.

D. $5/5=1$.




### 4. В примере логиты равны 2, 1 и 0, а softmax даёт примерно 0,665, 0,245 и 0,090. Какую проверку можно выполнить по этим числам?

A. Утверждать, что модель верно ответит на 66,5 % будущих снимков.

B. Утверждать, что вероятность ошибки равна 9 %.

C. Определить, есть ли нужный предмет в кадре.

D. Проверить, что оценки складываются примерно в 1.




### 5. Что следует проверить, если модель не выдала название изображённой на фото фигурки?

A. Цвет корпуса компьютера.

B. Есть ли подходящий класс в словаре модели.

C. Достаточно ли яркое окно ноутбука.

D. Было ли имя файла написано латиницей.




### 6. Зачем в коде используют `AutoImageProcessor` из комплекта весов?

A. Чтобы подготовить кадр способом, ожидаемым моделью.

B. Чтобы автоматически добавить новый класс.

C. Чтобы заменить веса на обучаемые.

D. Чтобы найти предмет и вырезать его из исходной фотографии.




### 7. Фотография содержит чашку и ложку, но классификатор вывел только три метки для всего кадра. Какого результата из этого вывода получить нельзя?

A. Три наиболее высоко оценённых класса.

B. Оценку каждого выведенного класса.

C. Индекс класса с наибольшей оценкой.

D. Координаты чашки и ложки.




### 8. На одном кадре класс «кружка» получил оценку 0,48, а на другом — 0,71. Что можно заключить без дополнительной проверки?

A. На втором фото кружка точно есть.

B. Модель верна ровно в 71 % подобных случаев.

C. Во втором запуске оценка этого класса выше.

D. Второй снимок обязательно легче для любого классификатора.




### 9. В таблице для пяти снимков верные метки попали в top-1 в трёх случаях и в top-3 в четырёх. Каковы соответствующие доли?

A. top-1 — 0,6; top-3 — 0,8.

B. top-1 — 0,8; top-3 — 0,6.

C. top-1 — 0,3; top-3 — 0,4.

D. top-1 — 3; top-3 — 4.




### 10. Что записать, чтобы другой человек мог повторить опыт классификации?

A. Только самый удачный ответ модели.

B. Идентификатор весов, версии библиотек, исходный файл и подготовку входа.

C. Только имя класса, который оказался первым.

D. Только снимок экрана с выводом.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Посчитайте значения в списке; каждое соответствует одному классу.



**Решение**


**B.** В списке три оценки, по одной на каждый из трёх классов.



### Вопрос 2

**Подсказка**


Softmax сохраняет порядок логитов.



**Решение**


**C.** Наибольший логит равен 2, поэтому класс A получит наибольшую оценку.



### Вопрос 3

**Подсказка**


В top-3 входят верные метки на первых, вторых и третьих местах.



**Решение**


**A.** Верная метка оказалась среди трёх первых в четырёх случаях из пяти: $4/5=0{,}8$.



### Вопрос 4

**Подсказка**


Softmax нормирует оценки внутри списка классов.



**Решение**


**D.** Сумма равна приблизительно 1; сами числа не дают оценку частоты верных ответов на новых данных.



### Вопрос 5

**Подсказка**


Классификатор выбирает метку из заданного набора.



**Решение**


**B.** Если словарь не содержит подходящей категории, модель выберет ближайший из доступных ей классов.



### Вопрос 6

**Подсказка**


При обучении изображения преобразовали в определённый формат.



**Решение**


**A.** Процессор задаёт нужные преобразования, включая изменение размера, обрезку и нормализацию.



### Вопрос 7

**Подсказка**


Классификатор описывает кадр целиком, а не отдельные места в нём.



**Решение**


**D.** Для положения объектов нужны координаты рамок, которые выдаёт детектор.



### Вопрос 8

**Подсказка**


Отделите сравнение двух чисел от вывода о правильности модели.



**Решение**


**C.** Можно сравнить сами оценки. Чтобы судить о точности на новых снимках, нужна отдельная размеченная проверка.



### Вопрос 9

**Подсказка**


Разделите число успехов каждого вида на общее число снимков.



**Решение**


**A.** Получаем $3/5=0{,}6$ для top-1 и $4/5=0{,}8$ для top-3.



### Вопрос 10

**Подсказка**


Для повтора понадобятся те же данные и условия обработки.



**Решение**


**B.** Веса модели, зависимости, файл и преобразования влияют на результат; их следует указать вместе.



